In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pandas tqdm Sastrawi

import pandas as pd
import re
from tqdm import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [3]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def clean_text_id(text):
    """
    Bersihin text bahasa Indonesia:
    - lowercase
    - hapus simbol / punctuation
    - hapus spasi berlebih
    - stemming
    """
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = stemmer.stem(text)
    return text

In [4]:
# ===== PATH =====
BASE_DIR = "/content/drive/MyDrive/capstonelb/"
EXCEL_LP = BASE_DIR + "LP and Course Mapping.xlsx"
EXCEL_RB = BASE_DIR + "Resource Data Learning Buddy.xlsx"

# ===== LOAD EXCEL =====
xls_lp = pd.ExcelFile(EXCEL_LP)
print("Sheet LP:", xls_lp.sheet_names)

xls_rb = pd.ExcelFile(EXCEL_RB)
print("Sheet Resource Buddy:", xls_rb.sheet_names)

# ===== Load sheet penting =====
df_tutorials = pd.read_excel(xls_lp, sheet_name="Tutorials")
df_questions = pd.read_excel(xls_rb, sheet_name="Current Interest Questions")
df_lp_mapping = pd.read_excel(xls_lp, sheet_name="LP + Course")
df_learning_path = pd.read_excel(xls_lp, sheet_name="Learning Path")
df_course = pd.read_excel(xls_lp, sheet_name="Course")
df_course_level = pd.read_excel(xls_lp, sheet_name="Course Level")
df_tutorials = pd.read_excel(xls_lp, sheet_name="Tutorials")

Sheet LP: ['LP + Course', 'Learning Path', 'Course', 'Tutorials', 'Course Level']
Sheet Resource Buddy: ['Learning Path Answer', 'Current Interest Questions', 'Current Tech Questions', 'Skill Keywords', 'Student Progress']


In [5]:
import requests

# ===== KONFIGURASI =====
BASE_URL = "https://jrkqcbmjknzgpbtrupxh.supabase.co/rest/v1"
API_KEY = "sb_publishable_h889CjrPIGwCMA9I4oTTaA_2L22Y__R"

headers = {
    "apikey": API_KEY,
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}


In [6]:
url_lp = f"{BASE_URL}/learning_paths"

res = requests.get(url_lp, headers=headers)
data_lp = res.json()

df_lp_api = pd.DataFrame(data_lp)

print(df_lp_api.head())


   learning_path_id             learning_path_name
0                 1                    AI Engineer
1                 2              Android Developer
2                 3  Back-End Developer JavaScript
3                 4      Back-End Developer Python
4                 5                 Data Scientist


In [7]:
url_courses = f"{BASE_URL}/courses"
res = requests.get(url_courses, headers=headers)
data_courses = res.json()

df_courses_api = pd.DataFrame(data_courses)
print(df_courses_api.head())


   course_id  learning_path_id                                   course_name  \
0          2                 1             Belajar Fundamental Deep Learning   
1          3                 1         Belajar Machine Learning untuk Pemula   
2          4                 1                      Machine Learning Terapan   
3          5                 1  Membangun Proyek Deep Learning Tingkat Mahir   
4          6                 1             Memulai Pemrograman dengan Python   

   course_level_str  
0                 3  
1                 2  
2                 4  
3                 4  
4                 1  


In [8]:
url_levels = f"{BASE_URL}/course_levels"
res = requests.get(url_levels, headers=headers)
data_levels = res.json()

df_levels_api = pd.DataFrame(data_levels)
print(df_levels_api.head())


   id course_level
0   1        Dasar
1   2       Pemula
2   3     Menengah
3   4        Mahir
4   5  Profesional


In [9]:
url_tutorials = f"{BASE_URL}/tutorials"
res = requests.get(url_tutorials, headers=headers)
data_tutorials = res.json()

df_tutorials_api = pd.DataFrame(data_tutorials)
print(df_tutorials_api.head())


   tutorial_id  course_id                                     tutorial_title  \
0            1          1                                       Taksonomi AI   
1            2          1  [Story] Machine Learning: Harapan menjadi keny...   
2            3          1                                    Rangkuman Kelas   
3            4          1                         Tipe-Tipe Machine Learning   
4            5          1                                      Forum Diskusi   

   type content  
0  None    None  
1  None    None  
2  None    None  
3  None    None  
4  None    None  


In [10]:
df_full = df_tutorials.merge(
    df_course, on="course_id", how="left"
).merge(
    df_learning_path, on="learning_path_id", how="left"
).merge(
    df_course_level,
    left_on="course_level_str",
    right_on="id",
    how="left"
)

print(df_full.columns)


Index(['tutorial_id', 'course_id', 'tutorial_title', 'Unnamed: 3',
       'Unnamed: 4', 'learning_path_id', 'course_name', 'course_level_str',
       'hours_to_study', 'learning_path_name', 'id', 'course_level'],
      dtype='object')


In [11]:
df_lp_mapping["lp_clean"] = df_lp_mapping["learning_path_name"].astype(str).apply(clean_text_id)

In [12]:
tqdm.pandas()
df_questions['clean_question'] = df_questions['question_desc'].progress_apply(clean_text_id)
df_tutorials['clean_tutorial'] = df_tutorials['tutorial_title'].progress_apply(clean_text_id)

100%|██████████| 7771/7771 [03:30<00:00, 37.00it/s]


In [13]:
print("=== Questions Sample ===")
display(df_questions.head())

print("=== Tutorials Sample ===")
display(df_tutorials.head())

=== Questions Sample ===


,question_desc,option_text,category,clean_question
0,Mana kegiatan yang paling relate denganmu di p...,Mencoba membuat menu sarapan,Mobile Development,mana giat yang paling relate dengan di pagi hari
1,Mana kegiatan yang paling relate denganmu di p...,Baca atau lihat info viral dari berbagai sumber,Artificial Intelligence,mana giat yang paling relate dengan di pagi hari
2,Mana kegiatan yang paling relate denganmu di p...,Membersihkan kamar,Cloud Computing,mana giat yang paling relate dengan di pagi hari
3,Mana kegiatan yang paling relate denganmu di p...,Coret-coret atau menulis di buku,Web Development,mana giat yang paling relate dengan di pagi hari
4,"Jika sedang menghadapi masalah, cara mana yang...",Coba berbagai cara sampai menemukan solusinya,Mobile Development,jika sedang hadap masalah cara mana yang palin...


=== Tutorials Sample ===


,tutorial_id,course_id,tutorial_title,Unnamed: 3,Unnamed: 4,clean_tutorial
0,1,1,Taksonomi AI,NaN,NaN,taksonomi ai
1,2,1,[Story] Machine Learning: Harapan menjadi keny...,NaN,NaN,story machine learning harap jadi nyata
2,3,1,Rangkuman Kelas,NaN,NaN,rangkum kelas
3,4,1,Tipe-Tipe Machine Learning,NaN,NaN,tipetipe machine learning
4,5,1,Forum Diskusi,NaN,NaN,forum diskusi


In [14]:
!pip install -U sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from tqdm import tqdm

In [15]:
questions_texts = df_questions['clean_question'].tolist()
tutorials_texts = df_tutorials['clean_tutorial'].tolist()

# ===== Hapus duplikat tutorial =====
df_tutorials = df_tutorials.drop_duplicates(subset=['clean_tutorial']).reset_index(drop=True)
tutorials_texts = df_tutorials['clean_tutorial'].tolist()

In [16]:
# Model cepat & multilingual, cocok bahasa Indonesia
model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [17]:
# ===== Encode semua text =====
tqdm.pandas()
print("Encoding questions...")
questions_embeddings = model.encode(questions_texts, show_progress_bar=True)
print("Encoding tutorials...")
tutorials_embeddings = model.encode(tutorials_texts, show_progress_bar=True)

print("Questions embeddings shape:", questions_embeddings.shape)
print("Tutorials embeddings shape:", tutorials_embeddings.shape)

Encoding questions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding tutorials...


Batches:   0%|          | 0/149 [00:00<?, ?it/s]

Questions embeddings shape: (20, 384)
Tutorials embeddings shape: (4763, 384)


In [18]:
embedding_dim = tutorials_embeddings.shape[1]  # dimensi embedding
index_tutorials = faiss.IndexFlatL2(embedding_dim)
index_tutorials.add(np.array(tutorials_embeddings, dtype='float32'))

print("Total tutorial di FAISS index:", index_tutorials.ntotal)


Total tutorial di FAISS index: 4763


In [19]:
# (lebih atas)
df_lp_mapping = pd.read_excel(xls_lp, sheet_name="LP + Course")
df_full = df_tutorials.merge(df_lp_mapping, on="tutorial_title", how="left")


In [20]:
def search_tutorial(query, top_k=5):
    query_clean = clean_text_id(query)
    query_emb = model.encode([query_clean]).astype("float32")

    # Ambil lebih banyak kandidat dulu
    D, I = index_tutorials.search(query_emb, top_k * 5)

    results = []
    seen_titles = set()

    for idx in I[0]:
        row = df_full.iloc[idx]
        title = row["tutorial_title"]

        if title in seen_titles:
            continue

        seen_titles.add(title)
        results.append(row)

        if len(results) == top_k:
            break

    return results

In [21]:
test_query = "mana kegiatan yang paling relate denganmu di pagi hari"
top_results = search_tutorial(test_query, top_k=5)
for i, r in enumerate(top_results):
    print(f"{i+1}. {r['tutorial_title']} (clean: {r['clean_tutorial']})")

1. Penanganan Overfitting dalam Klasifikasi Gambar (clean: tangan overfitting dalam klasifikasi gambar)
2. [Story] Ekstraksi Fitur! (clean: story ekstraksi fitur)
3. Library Pengolahan Data (clean: library olah data)
4. Latihan SKLearn SVR (clean: latih sklearn svr)
5. Dasar-Dasar Convolutional Neural Networks (CNNs) (clean: dasardasar convolutional neural networks cnns)


In [22]:
import pickle

# Path file simpanan
INDEX_PATH = BASE_DIR + "faiss_index_tutorials.bin"
TUTORIALS_PATH = BASE_DIR + "tutorials_df.pkl"

# Simpan FAISS index
faiss.write_index(index_tutorials, INDEX_PATH)
print("FAISS index disimpan di:", INDEX_PATH)

# Simpan dataframe tutorial (supaya urutan index-nya sama dengan FAISS)
with open(TUTORIALS_PATH, "wb") as f:
    pickle.dump(df_tutorials[["tutorial_id", "tutorial_title", "clean_tutorial"]], f)

print("Dataframe tutorial disimpan di:", TUTORIALS_PATH)


FAISS index disimpan di: /content/drive/MyDrive/capstonelb/faiss_index_tutorials.bin
Dataframe tutorial disimpan di: /content/drive/MyDrive/capstonelb/tutorials_df.pkl


In [23]:
import faiss
import pickle

INDEX_PATH = BASE_DIR + "faiss_index_tutorials.bin"
TUTORIALS_PATH = BASE_DIR + "tutorials_df.pkl"

# Load FAISS index
index_tutorials = faiss.read_index(INDEX_PATH)
print("FAISS index loaded, total vectors:", index_tutorials.ntotal)

# Load dataframe tutorial
with open(TUTORIALS_PATH, "rb") as f:
    df_tutorials = pickle.load(f)

print("df_tutorials loaded, shape:", df_tutorials.shape)


FAISS index loaded, total vectors: 4763
df_tutorials loaded, shape: (4763, 3)


In [24]:
test_query = "mana kegiatan yang paling relate denganmu di pagi hari"
top_results = search_tutorial(test_query, top_k=5)

for i, r in enumerate(top_results):
    print(f"{i+1}. {r['tutorial_title']} (clean: {r['clean_tutorial']})")


1. Penanganan Overfitting dalam Klasifikasi Gambar (clean: tangan overfitting dalam klasifikasi gambar)
2. [Story] Ekstraksi Fitur! (clean: story ekstraksi fitur)
3. Library Pengolahan Data (clean: library olah data)
4. Latihan SKLearn SVR (clean: latih sklearn svr)
5. Dasar-Dasar Convolutional Neural Networks (CNNs) (clean: dasardasar convolutional neural networks cnns)


In [25]:
def chatbot(user_input, top_k=5):
    results = search_tutorial(user_input, top_k=top_k)

    if len(results) == 0:
        return "Maaf, belum menemukan yang cocok. Coba pakai kata lain ya 😊"

    response = "✨ Rekomendasi terbaik untuk kamu:\n\n"

    for i, row in enumerate(results, 1):
        tutorial = row["tutorial_title"]
        lp = row.get("learning_path_name", "")
        course = row.get("course_name", "")
        level = row.get("course_level_str", "")

        response += f"{i}. **{tutorial}**\n"
        if pd.notna(lp):
            response += f"   • Learning Path : {lp}\n"
        if pd.notna(course):
            response += f"   • Course        : {course}\n"
        if pd.notna(level):
            response += f"   • Level         : {level}\n"
        response += "\n"

    return response

In [26]:
user_input = "aku mau belajar kebiasaan produktif di pagi hari"
print(chatbot(user_input, top_k=5))


✨ Rekomendasi terbaik untuk kamu:

1. **[Story] Ekstraksi Fitur!**
   • Learning Path : AI Engineer
   • Course        : Belajar Fundamental Deep Learning
   • Level         : Menengah

2. **Model Sekuensial dengan Beberapa Layer**
   • Learning Path : MLOps Engineer
   • Course        : Belajar Fundamental Deep Learning
   • Level         : Menengah

3. **Latihan Kuis**
   • Learning Path : Google Cloud Professional
   • Course        : Menjadi Google Cloud Architect
   • Level         : Mahir

4. **Mengenal String Lebih Dalam**
   • Learning Path : Android Developer
   • Course        : Memulai Pemrograman dengan Kotlin
   • Level         : Dasar

5. **Latihan Studi Kasus: Membuat Model di Movie Go**
   • Learning Path : Back-End Developer Python
   • Course        : Belajar Fundamental Back-End dengan Python
   • Level         : Menengah




In [27]:
df_full.head()


,tutorial_id,course_id,tutorial_title,Unnamed: 3,Unnamed: 4,clean_tutorial,learning_path_name,course_name,course_level_str
0,1,1,Taksonomi AI,NaN,NaN,taksonomi ai,AI Engineer,Belajar Dasar AI,Dasar
1,1,1,Taksonomi AI,NaN,NaN,taksonomi ai,Gen AI Engineer,Belajar Dasar AI,Dasar
2,1,1,Taksonomi AI,NaN,NaN,taksonomi ai,MLOps Engineer,Belajar Dasar AI,Dasar
3,2,1,[Story] Machine Learning: Harapan menjadi keny...,NaN,NaN,story machine learning harap jadi nyata,AI Engineer,Belajar Dasar AI,Dasar
4,2,1,[Story] Machine Learning: Harapan menjadi keny...,NaN,NaN,story machine learning harap jadi nyata,Gen AI Engineer,Belajar Dasar AI,Dasar


In [28]:
df_interest = pd.read_excel(xls_rb, sheet_name="Current Interest Questions")
df_tech = pd.read_excel(xls_rb, sheet_name="Current Tech Questions")


In [29]:
df_interest["clean_question"] = df_interest["question_desc"].apply(clean_text_id)
df_tech["clean_question"] = df_tech["question_desc"].apply(clean_text_id)


In [30]:
interest_texts = df_interest["clean_question"].tolist()
tech_texts = df_tech["clean_question"].tolist()

interest_embeddings = np.array(model.encode(interest_texts), dtype="float32")
tech_embeddings = np.array(model.encode(tech_texts), dtype="float32")


In [31]:
index_interest = faiss.IndexFlatL2(interest_embeddings.shape[1])
index_tech = faiss.IndexFlatL2(tech_embeddings.shape[1])

index_interest.add(interest_embeddings)
index_tech.add(tech_embeddings)


In [32]:
def search_interest(query, top_k=3):
    query_clean = clean_text_id(query)
    emb = np.array(model.encode([query_clean]), dtype="float32")

    D, I = index_interest.search(emb, top_k)

    results = []
    for idx in I[0]:
        results.append({
            "question": df_interest["question_desc"].iloc[idx],
            "category": df_interest["category"].iloc[idx]
        })
    return results


In [33]:
def search_tech(query, top_k=3):
    query_clean = clean_text_id(query)
    emb = np.array(model.encode([query_clean]), dtype="float32")

    D, I = index_tech.search(emb, top_k)

    results = []
    for idx in I[0]:
        results.append({
            "question": df_tech["question_desc"].iloc[idx],
            "tech_category": df_tech["tech_category"].iloc[idx],
            "difficulty": df_tech["difficulty"].iloc[idx]
        })
    return results


In [34]:
def detect_intent_v2(user_input, threshold=0.7):
    query_clean = clean_text_id(user_input)
    emb = np.array(model.encode([query_clean]), dtype="float32")

    d_tut, _ = index_tutorials.search(emb, 1)
    d_int, _ = index_interest.search(emb, 1)
    d_tech, _ = index_tech.search(emb, 1)

    scores = {
        "tutorial": float(d_tut[0][0]),
        "interest": float(d_int[0][0]),
        "tech": float(d_tech[0][0])
    }

    best_intent = min(scores, key=scores.get)
    best_score = scores[best_intent]

    if best_score > threshold:
        return "tutorial", scores

    return best_intent, scores


In [35]:
import random

FRIENDLY_OPENERS = [
    "Wah, menarik nih yang kamu bahas! 😄",
    "Okeee, aku bantu ya!",
    "Asik, ini topik yang bagus banget buat dipelajari ✨",
    "Nice question! Yuk kita bahas bareng 🤝"
]

RECOMMENDATION_INTROS = [
    "Ini beberapa materi yang menurutku pas banget buat kamu:",
    "Aku nemuin ini buat kamu, coba cek ya:",
    "Menurutku ini cocok buat kamu pelajari selanjutnya:",
    "Coba mulai dari ini dulu ya:"
]


In [36]:
def chatbot(user_input, top_k=5):
    intent, scores = detect_intent_v2(user_input)

    response = ""
    response += random.choice(FRIENDLY_OPENERS) + "\n\n"

    if intent == "interest":
        data = search_interest(user_input, 1)[0]
        response += f"Kayaknya kamu tertarik di area **{data['category']}**.\n\n"

    elif intent == "tech":
        data = search_tech(user_input, 1)[0]
        response += (
            f"Kamu lagi bahas topik **{data['tech_category']}** "
            f"(Level: {data['difficulty']}).\n\n"
        )

    response += random.choice(RECOMMENDATION_INTROS) + "\n\n"

    tutorials = search_tutorial(user_input, top_k)
    for i, t in enumerate(tutorials, 1):
        response += f"{i}. {t['tutorial_title']}\n"

    response += "\nKalau mau bahas topik lain, tanya aja ya! 😊"

    return response


In [37]:
print(chatbot("aku mau belajar kebiasaan produktif pagi hari"))


Nice question! Yuk kita bahas bareng 🤝

Menurutku ini cocok buat kamu pelajari selanjutnya:

1. [Story] Ekstraksi Fitur!
2. Model Sekuensial dengan Beberapa Layer
3. Latihan Kuis
4. Mengenal String Lebih Dalam
5. [Story] Beda Algoritma Beda Juga Cara Kerjanya? Kok Bisa?

Kalau mau bahas topik lain, tanya aja ya! 😊


In [38]:
LEVEL_ORDER = {
    "Dasar": 1,
    "Menengah": 2,
    "Mahir": 3
}

def get_level_value(level):
    return LEVEL_ORDER.get(level, 1)


In [39]:
def generate_roadmap(skill, user_level):
    """
    skill: string (misal: 'javascript', 'react')
    user_level: 'Dasar' | 'Menengah' | 'Mahir'
    """

    # Filter tutorial yang mengandung skill
    filtered = df_full[
        df_full["tutorial_title"].str.lower().str.contains(skill, na=False)
    ].copy()

    if filtered.empty:
        return f"Tidak ditemukan roadmap untuk skill: {skill}"

    # Tambah ranking level
    filtered["level_rank"] = filtered["course_level_str"].apply(get_level_value)
    user_rank = get_level_value(user_level)

    # Ambil roadmap yang levelnya >= level user
    roadmap = filtered[filtered["level_rank"] >= user_rank]

    roadmap = roadmap.sort_values("level_rank")

    results = []
    for _, row in roadmap.iterrows():
        results.append({
            "tutorial": row["tutorial_title"],
            "course": row.get("course_name", ""),
            "learning_path": row.get("learning_path_name", ""),
            "level": row.get("course_level_str", "")
        })

    return results


In [40]:
print(generate_roadmap("javascript", "Dasar"))
print(generate_roadmap("react", "Menengah"))


[{'tutorial': 'Latihan: Menjalankan JavaScript Menggunakan Node.js', 'course': 'Belajar Back-End Pemula dengan JavaScript', 'learning_path': 'Back-End Developer JavaScript', 'level': 'Pemula'}, {'tutorial': 'Dasar JavaScript', 'course': 'Belajar Dasar Pemrograman Web', 'learning_path': 'Front-End Web Developer', 'level': 'Dasar'}, {'tutorial': 'Dasar JavaScript', 'course': 'Belajar Dasar Pemrograman Web', 'learning_path': 'React Developer', 'level': 'Dasar'}, {'tutorial': 'Menuliskan Syntax Javascript pada HTML', 'course': 'Belajar Dasar Pemrograman Web', 'learning_path': 'Front-End Web Developer', 'level': 'Dasar'}, {'tutorial': 'Menuliskan Syntax Javascript pada HTML', 'course': 'Belajar Dasar Pemrograman Web', 'learning_path': 'React Developer', 'level': 'Dasar'}, {'tutorial': 'Membuat Berkas JavaScript Pertama', 'course': 'Belajar Dasar Pemrograman Web', 'learning_path': 'Front-End Web Developer', 'level': 'Dasar'}, {'tutorial': 'Membuat Berkas JavaScript Pertama', 'course': 'Belaj

In [41]:
def build_role_skill_map():
    role_skill_map = {}

    # Ambil semua learning path (anggap sbg role)
    roles = df_learning_path["learning_path_name"].dropna().unique()

    for role in roles:
        # Ambil courses yang milik role ini
        courses = df_lp_mapping[
            df_lp_mapping["learning_path_name"] == role
        ]["course_name"].dropna().unique()

        # Ambil tutorial titles yang ada di course tsb
        skills = df_tutorials[
            df_tutorials["course_name"].isin(courses)
        ]["clean_tutorial"].dropna().unique()

        role_skill_map[role.lower()] = list(skills)

    return role_skill_map

In [42]:
def learning_buddy(user_input, role, skill, level):
    roadmap = generate_roadmap(skill, level)

    response = f"Halo! 😊\n"
    response += f"Kamu role: {role}\n"
    response += f"Skill yang mau dikembangkan: {skill}\n\n"

    response += "📚 Roadmap belajar buat kamu:\n\n"

    if isinstance(roadmap, str):
        response += roadmap
    else:
        for i, r in enumerate(roadmap, 1):
            response += (
                f"{i}. {r['tutorial']}\n"
                f"   Course : {r['course']}\n"
                f"   Path   : {r['learning_path']}\n"
                f"   Level  : {r['level']}\n\n"
            )

    return response


In [43]:
def build_role_skill_map():
    role_skill_map = {}

    # anggap learning_path_name sebagai "role"
    roles = df_lp_mapping["learning_path_name"].dropna().unique()

    for role in roles:
        subset = df_lp_mapping[df_lp_mapping["learning_path_name"] == role]

        # ambil tutorial_title sebagai "skills"
        skills_raw = subset["tutorial_title"].dropna().astype(str)

        # kalau kamu sudah punya clean_text_id, boleh dibersihin
        skills_clean = skills_raw.apply(clean_text_id).unique().tolist()

        # simpan dengan role yang sudah dibersihin
        role_key = clean_text_id(role)
        role_skill_map[role_key] = skills_clean

    return role_skill_map

In [44]:
print(
    learning_buddy(
        user_input="aku mau jago frontend",
        role="frontend developer",
        skill="javascript",
        level="Dasar"
    )
)


Halo! 😊
Kamu role: frontend developer
Skill yang mau dikembangkan: javascript

📚 Roadmap belajar buat kamu:

1. Latihan: Menjalankan JavaScript Menggunakan Node.js
   Course : Belajar Back-End Pemula dengan JavaScript
   Path   : Back-End Developer JavaScript
   Level  : Pemula

2. Dasar JavaScript
   Course : Belajar Dasar Pemrograman Web
   Path   : Front-End Web Developer
   Level  : Dasar

3. Dasar JavaScript
   Course : Belajar Dasar Pemrograman Web
   Path   : React Developer
   Level  : Dasar

4. Menuliskan Syntax Javascript pada HTML
   Course : Belajar Dasar Pemrograman Web
   Path   : Front-End Web Developer
   Level  : Dasar

5. Menuliskan Syntax Javascript pada HTML
   Course : Belajar Dasar Pemrograman Web
   Path   : React Developer
   Level  : Dasar

6. Membuat Berkas JavaScript Pertama
   Course : Belajar Dasar Pemrograman Web
   Path   : Front-End Web Developer
   Level  : Dasar

7. Membuat Berkas JavaScript Pertama
   Course : Belajar Dasar Pemrograman Web
   Path   :

In [45]:
ROLE_SKILL_MAP = build_role_skill_map()
print(list(ROLE_SKILL_MAP.keys())[:5])

['ai engineer', 'android developer', 'backend developer javascript', 'backend developer python', 'data scientist']


In [46]:
SKILL_KEYWORDS = {
    "frontend": ["html", "css", "javascript", "react", "vue", "dom"],
    "backend": ["api", "database", "sql", "docker", "kubernetes", "node", "django"],
    "ai": ["machine learning", "deep learning", "neural", "tensorflow", "pytorch"],
    "devops": ["docker", "kubernetes", "ci/cd", "pipeline", "cloud"]
}


In [47]:
# def extract_skills_from_text(user_text, role):
#     text_lower = user_text.lower()
#     text_clean = clean_text_id(user_text)

#     if not role:
#         return []

#     role_clean = clean_text_id(role)

#     # cari role yang cocok di ROLE_SKILL_MAP
#     matched_role = None
#     for k in ROLE_SKILL_MAP.keys():
#         if role_clean in clean_text_id(k) or clean_text_id(k) in role_clean:
#             matched_role = k
#             break

#     # kalau masih ga nemu, fallback pakai keyword manual
#     if not matched_role:
#         fallback = []
#         for group in SKILL_KEYWORDS.values():
#             for kw in group:
#                 if kw in text_lower or kw in text_clean:
#                     fallback.append(kw)
#         return list(set(fallback))

#     # extract skills dari dataset
#     skills = ROLE_SKILL_MAP.get(matched_role, [])
#     found = []

#     for s in skills:
#         s_clean = clean_text_id(s)
#         if s_clean in text_clean or s.lower() in text_lower:
#             found.append(s)

#     return list(set(found))


In [48]:
def classify_skill_level(text):
    text = text.lower()

    if any(w in text for w in ["pemula", "baru mulai", "dasar"]):
        return "Beginner"
    elif any(w in text for w in ["cukup paham", "menengah", "intermediate"]):
        return "Intermediate"
    elif any(w in text for w in ["mahir", "expert", "lanjut"]):
        return "Advanced"
    else:
        return "Unknown"

In [49]:
print(classify_skill_level("aku masih pemula javascript"))
print(classify_skill_level("aku sudah cukup paham react"))
print(classify_skill_level("aku expert di machine learning"))


Beginner
Intermediate
Advanced


In [50]:
def detect_role_from_text(user_text):
    text = clean_text_id(user_text)

    for role in ROLE_SKILL_MAP.keys():
        if role in text:
            return role

    return "unknown"

In [51]:
print(detect_role_from_text("aku pengen jadi frontend developer"))

unknown


In [52]:
def skill_assessment_engine(user_input):
    role = detect_role_from_text(user_input)
    skills = extract_skills_from_text(user_input)
    level = classify_skill_level(user_input)

    return {
        "role": role,
        "skills": skills,
        "level": level
    }

In [53]:
ROLE_SYNONYMS = {
    # key = canonical role (pakai nama learning path dari dataset, tapi dibersihin dulu)
    clean_text_id("Frontend Developer"): [
        "frontend developer", "front end developer", "frontend", "front-end", "web developer", "developer web"
    ],
    clean_text_id("Back-End Developer Python"): [
        "backend developer", "back end developer", "back-end", "backend python", "back end python", "server side"
    ],
    clean_text_id("Android Developer"): [
        "android developer", "developer android", "mobile developer", "android dev"
    ],
    clean_text_id("AI Engineer"): [
        "ai engineer", "machine learning engineer", "ml engineer", "ai dev", "ai developer"
    ],
    clean_text_id("Data Scientist"): [
        "data scientist", "data science"
    ],
}

In [54]:
def detect_role_v2(user_input):
    """
    Role classifier v2:
    1) Cek sinonim role eksplisit di text
    2) Kalau tidak ketemu, coba tebak dari skill yang disebut
    3) Kalau masih tidak yakin, pakai embedding similarity ke role descriptions
    """
    text_clean = clean_text_id(user_input)

    # 1. Langkah pertama: cari role eksplisit via sinonim
    for canonical_role, patterns in ROLE_SYNONYMS.items():
        for p in patterns:
            if p in text_clean:
                return canonical_role  # sudah dalam bentuk clean_text_id

    # 2. Langkah kedua: voting berdasarkan skill (ROLE_SKILL_MAP)
    # ROLE_SKILL_MAP: {clean_role_name: [list clean_tutorials]}
    best_role = None
    best_score = 0

    for role, skills in ROLE_SKILL_MAP.items():
        score = 0
        for sk in skills:
            # kalau nama tutorial/skill muncul di input
            if sk in text_clean:
                score += 1
        if score > best_score:
            best_score = score
            best_role = role

    if best_role is not None and best_score > 0:
        return best_role

    # 3. Langkah ketiga: fallback embedding similarity ke deskripsi role
    # Bangun text per role: "role + beberapa skill"
    role_texts = []
    role_names = []
    for role, skills in ROLE_SKILL_MAP.items():
        combined = role + " " + " ".join(skills[:20])  # ambil max 20 skill biar ga kepanjangan
        role_texts.append(combined)
        role_names.append(role)

    # Encode role + query
    role_emb = model.encode(role_texts)
    role_emb = np.array(role_emb, dtype="float32")

    query_emb = model.encode([text_clean])
    query_emb = np.array(query_emb, dtype="float32")[0]

    # cosine similarity
    # cos_sim = (A·B) / (|A||B|)
    norms = np.linalg.norm(role_emb, axis=1) * np.linalg.norm(query_emb)
    sims = role_emb.dot(query_emb) / (norms + 1e-8)

    best_idx = int(np.argmax(sims))
    best_sim = float(sims[best_idx])
    best_role_embed = role_names[best_idx]

    # threshold boleh kamu adjust
    if best_sim > 0.35:
        return best_role_embed

    # 4. Kalau semua gagal
    return "unknown"

In [55]:
def build_role_skill_map_from_dataset():
    role_skill = {}

    for lp in df_lp_mapping["learning_path_name"].dropna().unique():
        clean_lp = clean_text_id(lp)

        tutorials = df_lp_mapping[
            df_lp_mapping["learning_path_name"] == lp
        ]["tutorial_title"].dropna().unique().tolist()

        # clean tutorial titles
        tutorials = [clean_text_id(t) for t in tutorials]

        role_skill[clean_lp] = tutorials

    return role_skill


ROLE_SKILL_MAP = build_role_skill_map_from_dataset()


In [56]:
print(list(ROLE_SKILL_MAP.keys())[:10])
print(ROLE_SKILL_MAP.get(clean_text_id("Front-End Web Developer"), [])[:10])


['ai engineer', 'android developer', 'backend developer javascript', 'backend developer python', 'data scientist', 'devops engineer', 'frontend web developer', 'gen ai engineer', 'google cloud professional', 'ios developer']
['nal dengan functional programming', 'return value', 'uji akhir', 'kuis coding sang atur program', 'struktur data dengan object', 'tangan error dengan callback', 'apa itu promise', 'kenal tangan eror', 'tuju hak cipta', 'rangkum materi']


In [57]:
def extract_skills_from_text(user_text, role):
    text = user_text.lower()
    found = []

    # ✅ PAKAI SKILL_KEYWORDS (yang sudah kamu punya)
    for group in SKILL_KEYWORDS.values():
        for kw in group:
            if kw in text:
                found.append(kw)

    # ✅ fallback kalau kosong, biar nggak blank total
    if not found and role in ROLE_SKILL_MAP:
        sample = ROLE_SKILL_MAP[role][:5]
        found.extend(sample)

    return list(set(found))


In [71]:
def skill_assessment_engine_v2(user_input):
    role = detect_role_v2(user_input)
    level = classify_skill_level(user_input)

    skills = extract_skills_from_text(user_input, role)

    return {
        "role": role,
        "skills": skills,
        "level": level
    }


In [72]:
print(skill_assessment_engine_v2("aku mau jadi frontend developer, masih pemula di html dan css"))

print(skill_assessment_engine_v2("lagi belajar kubernetes dan docker buat backend, udah cukup paham sih"))

print(skill_assessment_engine_v2("pengen jadi ai engineer, masih pemula banget di machine learning"))


{'role': 'frontend developer', 'skills': ['html', 'css'], 'level': 'Beginner'}
{'role': 'mlops engineer', 'skills': ['kubernetes', 'docker'], 'level': 'Intermediate'}
{'role': 'ai engineer', 'skills': ['machine learning'], 'level': 'Beginner'}


In [73]:
# Bikin kolom learning path yang sudah dibersihkan (clean) untuk matching role
df_lp_mapping["lp_clean"] = df_lp_mapping["learning_path_name"].astype(str).apply(clean_text_id)


In [74]:
LEVEL_MAP_TO_DATASET = {
    "Beginner": ["Dasar"],
    "Intermediate": ["Menengah"],
    "Advanced": ["Mahir"],
    "Unknown": ["Dasar", "Menengah"]  # fallback kalau level gak ketebak
}

LEVEL_ORDER_DATASET = {
    "Dasar": 1,
    "Menengah": 2,
    "Mahir": 3
}

In [75]:
def generate_roadmap_from_data(role, level, max_weeks=4):
    """
    Generate roadmap belajar berdasarkan:
      - role (dari detect_role_v2 → bentuk clean_text_id)
      - level (Beginner / Intermediate / Advanced / Unknown)
      - df_lp_mapping (LP + Course dari Excel)

    Return: list string ["Week 1: ...", "Week 2: ...", ...]
    """

    # Kalau role belum ketemu sama sekali
    if role is None or role == "unknown":
        return [
            "Belum bisa menentukan roadmap karena role kamu belum jelas.",
            "Coba jelaskan role yang kamu mau, misalnya: frontend developer, backend, ai engineer, dll."
        ]

    # Tentukan level-level course yang cocok di dataset
    target_levels = LEVEL_MAP_TO_DATASET.get(level, ["Dasar"])

    subset = df_lp_mapping[
        (df_lp_mapping["lp_clean"] == role) &
        (df_lp_mapping["course_level_str"].isin(target_levels))
    ].copy()

    # Kalau filter dengan level kosong, fallback: abaikan level
    if subset.empty:
        subset = df_lp_mapping[df_lp_mapping["lp_clean"] == role].copy()

    if subset.empty:
        return [f"Belum ada course yang cocok di dataset untuk role: {role}"]

    # Urutkan berdasarkan level (Dasar < Menengah < Mahir)
    subset["level_rank"] = subset["course_level_str"].map(
        lambda x: LEVEL_ORDER_DATASET.get(x, 99)
    )

    subset = subset.sort_values(["level_rank", "course_name"])

    # Ambil course unik supaya tidak dobel-dobel
    subset_courses = subset[["course_name", "course_level_str"]].drop_duplicates()

    roadmap = []
    for i, (_, row) in enumerate(subset_courses.iterrows(), 1):
        if i > max_weeks:
            break
        course_name = row["course_name"]
        level_str = row["course_level_str"]
        roadmap.append(f"Week {i}: {course_name} (Level: {level_str})")

    return roadmap

In [76]:
def assess_and_generate_roadmap(user_input, max_weeks=4):
    """
    1) Analisis input user → role, skills, level  (pakai skill_assessment_engine_v2)
    2) Generate roadmap belajar dari df_lp_mapping
    """
    assess = skill_assessment_engine_v2(user_input)

    role = assess["role"]
    level = assess["level"]
    skills = assess["skills"]

    roadmap = generate_roadmap_from_data(role, level, max_weeks=max_weeks)

    return {
        "role": role,
        "level": level,
        "skills": skills,
        "roadmap": roadmap
    }

In [77]:
result = assess_and_generate_roadmap(
    "aku mau jadi frontend developer tapi aku masih pemula"
)

print("Role  :", result["role"])
print("Level :", result["level"])
print("Skills:", result["skills"])
print("\nRoadmap:")
for line in result["roadmap"]:
    print("-", line)


Role  : frontend developer
Level : Beginner
Skills: ['api']

Roadmap:
- Belum ada course yang cocok di dataset untuk role: frontend developer


In [78]:
def chatbot(user_input, top_k=5):
    assess = assess_and_generate_roadmap(user_input)

    # salam + penjelasan
    response = random.choice(FRIENDLY_OPENERS) + "\n\n"
    if assess["role"] != "unknown":
        response += f"Kamu cocok di jalur: **{assess['role']}** (level: {assess['level']})\n\n"

    # rekomendasi tutorial (FAISS)
    response += "Rekomendasi materi yang bisa kamu mulai:\n"
    ...

    # roadmap
    response += "\n📅 Roadmap belajar buat kamu:\n"
    for line in assess["roadmap"]:
        response += f"- {line}\n"

    return response


In [79]:
print(skill_assessment_engine_v2(
    "aku mau jadi frontend developer, masih pemula di html dan css"
))

print(skill_assessment_engine_v2(
    "lagi belajar docker dan kubernetes buat backend"
))

print(skill_assessment_engine_v2(
    "pengen jadi ai engineer, masih pemula di machine learning"
))


{'role': 'frontend developer', 'skills': ['html', 'css'], 'level': 'Beginner'}
{'role': 'mlops engineer', 'skills': ['kubernetes', 'docker'], 'level': 'Unknown'}
{'role': 'ai engineer', 'skills': ['machine learning'], 'level': 'Beginner'}
